# CALE Family Psychometrics: CFA, Multi-Group, And Validity Checks

This notebook tests whether CALE-family diagnostic indicators provide preliminary psychometric-style evidence: interpretable internal structure, robustness across groups, convergent validity with construct-relevant signals, and discriminant validity from construct-irrelevant nuisance signals.

本 notebook 不是为了做 leaderboard，而是为了检验 CALE 家族的 diagnostic indicators 是否有初步 psychometrics-style validity evidence：内部结构、跨组稳健性、与 construct-relevant signals 收敛、并且不被 construct-irrelevant nuisance signals 主导。

Interpretation note: `semopy` fits a continuous-indicator CFA approximation. CALE indicators are bounded and often binary, so all CFA results should be treated as preliminary internal-structure evidence rather than definitive ordinal CFA validation.

解释提醒：这些结果不能写成已经完成严格 psychometric validation；更稳妥的表述是 preliminary internal-structure and validity evidence。


In [ ]:
# Uncomment on the server if semopy is not installed.
# %pip install semopy pandas numpy

from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
from semopy import Model, calc_stats

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

## Psychometric Questions / 心理测量问题

Run this notebook with these questions in mind:

1. **Internal structure**: Is CALE behavior better represented by multiple related dimensions than by one undifferentiated score?  
   CALE behavior 是不是不只是一个总分，而是更适合被解释为多个相关但可区分的 latent dimensions？

2. **CALE-family comparison**: Do `generic_cale`, `attack_aware_cale`, and `full_attack_aware_cale` show similar measurement structure, or does each variant behave like a different instrument?  
   CALE 家族内部的不同 variant 是共享稳定结构，还是每个 variant 都像不同测量工具？

3. **Robustness/generalization**: Does the structure remain similar across target models and evaluator backends?  
   这个结构是否在 Qwen/Llama target split 和不同 evaluator backend 下保持相似？

4. **Convergent validity**: Do indicators intended to measure the same construct load together and correlate with construct-relevant signals?  
   理论上属于同一 construct 的 indicators 是否真的聚在一起，并且和 construct-relevant signals 收敛？

5. **Discriminant validity**: Are latent factors related but not redundant, and are they not mainly explained by nuisance signals such as target identity, reference label, domain, or framing style?  
   latent factors 是否相关但不冗余，并且不主要由 target identity、reference label、domain、framing style 这类 nuisance signals 解释？

6. **Claim boundary**: What can we safely claim without turning measurement evidence into a leaderboard?  
   我们能安全地说什么，而不把 measurement evidence 误写成排行榜？


## What To Look At / 指标怎么看

| Output | What it means | Good sign | If good, we can say |
|---|---|---|---|
| CFI / TLI | Approximate comparative fit | Higher is better; around .90 acceptable, .95 strong | The hypothesized structure fits better than weak/null baseline |
| RMSEA | Approximate residual misfit per df | Lower is better; < .08 acceptable, < .06 strong | Model misfit is not too large |
| SRMR | Average standardized residual | Lower is better; < .08 usually acceptable | Reproduced correlations are close to observed correlations |
| AIC / BIC | Model comparison criteria | Lower is better among competing models | One model is more parsimonious / better supported than another |
| Standardized loadings | Indicator-factor relation | > .70 strong, .40-.70 moderate, < .30 weak | Indicators meaningfully reflect intended latent factor |
| Factor correlations | Relation between latent factors | .30-.80 often useful; > .90 may be too high | Factors are related but distinguishable |

中文速读：

- `three_factor` 明显优于 `one_factor`：可以说 CALE behavior is not merely a single score。
- 大多数 intended loadings 中高：可以说 indicators align with theorized dimensions。
- factor correlations 不接近 1：可以说 dimensions are related but empirically distinguishable。
- Qwen/Llama/variant robustness 相似：可以说 internal structure is not driven by a single target model or evaluator variant。
- 如果 fit 很差或 loading 很弱：不要强 claim；改说 current indicators need refinement 或 structure remains exploratory。

## Configure Analysis Cases And Paths

Set `ANALYSIS_BACKEND` to choose the behavior matrix CSV for the primary single-case CFA cells. The default is `heuristic_full`, because it is the main CALE-family psychometrics input.

用 `ANALYSIS_BACKEND` 选择主 CFA 输入。默认是 `heuristic_full`，因为它是当前 CALE family psychometrics 的主数据。

Use DeepSeek or Qwen2.5-7B paths only for backend-robustness checks, not as the default family CFA.


## CFA Input Choice / CFA 输入选择

The default CFA input is **`heuristic_full`**, because it is the only current full-sample behavior matrix with both target models and the full CALE family ladder.

默认主 CFA 输入是 **`heuristic_full`**，因为它是目前唯一同时覆盖两个 target models 和完整 CALE family ladder 的 full-sample behavior matrix。

| Option | What it analyzes | What it can support |
|---|---|---|
| `heuristic_full` | Full heuristic/rule-based behavior matrix with all main variants | Primary CALE-family psychometrics: per-variant CFA, PCA/CFA structure, target split robustness |
| `qwen25_7b_limit1000` | Qwen2.5-7B local evaluator subset | Backend robustness for `direct_llm_judge` vs `full_attack_aware_cale`; not full CALE-family comparison |
| `deepseek_v4_pro_limit1000` | DeepSeek V4-Pro API evaluator subset | Backend robustness for `direct_llm_judge` vs `full_attack_aware_cale`; currently Qwen-target-only unless Llama chunk is added |
| `deepseek_v4_pro_limit20` | DeepSeek smoke-test subset | Pipeline sanity check only; too small for substantive CFA claims |

中文判断：

- 主 CFA / CALE family psychometrics 用 `heuristic_full`。
- DeepSeek/Qwen2.5-7B 是 evaluator-backend robustness，不替代主 CFA。
- 如果 strong evaluator 只包含 `direct_llm_judge` 和 `full_attack_aware_cale`，它不能回答 generic vs attack-aware vs full 的家族差异。


## Variant Choice / CALE 版本要不要分开？

Yes. `generic_cale`, `attack_aware_cale`, and `full_attack_aware_cale` should be inspected separately when available.

是的，应该分开看。原因是这些 variants 不是普通重复样本，而是不同 evaluator design / measurement procedure。如果把它们全混在一起，只能回答一个 pooled question：整体 CALE-family indicators 有没有共同结构。它不能说明每个 variant 自己的结构是否稳定。

Recommended reading / 推荐判断顺序：

1. **Pooled CALE-family CFA**: first ask whether all CALE-style rows share a broad internal structure.  
   先看总体 CALE family 有没有 multidimensional structure。

2. **Variant-specific CFA**: repeat the same model for `generic_cale`, `attack_aware_cale`, and `full_attack_aware_cale`.  
   再分别看每个 variant 的 fit/loadings 是否类似。

3. **If variant-specific results are similar**: claim structure is robust across CALE variants.  
   如果相似，可以说 internal structure is broadly stable across CALE variants。

4. **If variant-specific results differ**: claim variant changes the measurement structure.  
   如果差异很大，不要强行 pooled claim；应该说 attack-aware design changes diagnostic organization。

Important: some strong-evaluator runs may only include `direct_llm_judge` and `full_attack_aware_cale`. In that case, DS data cannot answer generic-vs-attack-vs-full CFA differences unless you rerun DS with all CALE variants.

注意：有些 strong evaluator run 可能只包含 `direct_llm_judge` 和 `full_attack_aware_cale`。如果 DS 文件里没有 `generic_cale` / `attack_aware_cale`，那这个 DS 数据本身不能比较三种 CALE variants，只能分析 DS + full_attack_aware_cale 的 CFA structure。

In [ ]:
# Choose the behavior matrix you want to analyze.
# Current default: CALE-family psychometrics on the heuristic full matrix.
# If your server output path differs, either edit DATA_PATHS below or set
# DATA_PATH manually after this cell.

DATA_PATHS = {
    "heuristic_full": Path("outputs/small_models_all/fever_dev_qwen25_15b_llama32_1b_neutral_full_eval_behavior_matrix.csv"),
    "qwen25_7b_limit1000": Path("outputs/small_models_all/fever_dev_qwen25_15b_llama32_1b_neutral_strong_qwen25_7b_limit1000_eval_behavior_matrix.csv"),
    "deepseek_v4_pro_limit1000": Path("outputs/small_models_all/fever_dev_qwen25_15b_llama32_1b_neutral_deepseek_v4_pro_limit1000_eval_behavior_matrix.csv"),
    "deepseek_v4_pro_limit20": Path("outputs/small_models_all/fever_dev_qwen25_15b_llama32_1b_neutral_deepseek_v4_pro_limit20_eval_behavior_matrix.csv"),
}

BACKEND_NOTES = {
    "heuristic_full": "Primary CALE-family psychometrics input: full sample, two target models, full variant ladder.",
    "qwen25_7b_limit1000": "Qwen2.5-7B strong evaluator subset. Use for backend robustness, not full CALE-family comparison.",
    "deepseek_v4_pro_limit1000": "DeepSeek V4-Pro strong evaluator subset. Use for backend robustness; current limit1000 may be Qwen-target-only.",
    "deepseek_v4_pro_limit20": "DeepSeek smoke-test CFA. Use only to debug the pipeline; sample is too small for substantive CFA claims.",
}

ANALYSIS_BACKEND = "heuristic_full"
DATA_PATH = DATA_PATHS[ANALYSIS_BACKEND]
OUTPUT_DIR = Path("figures/cfa_behavior_model/family_psychometrics") / ANALYSIS_BACKEND
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Candidate CALE variants. The notebook will keep whichever of these are
# actually present in the selected DATA_PATH.
CALE_VARIANTS = ["generic_cale", "attack_aware_cale", "full_attack_aware_cale"]

ITEMS = [
    "misinformation_detection",
    "framing_resistance",
    "claim_status_recognition",
    "error_rejection",
    "correction_accuracy",
    "evidence_grounding",
    "source_faithfulness",
    "hallucination_control",
    "uncertainty_handling",
]

EXCLUDED_PRIMARY_CFA_COLUMNS = [
    "final_score",
    "quality_label",
    "uncertainty",
    "reference_is_nei",
    "reference_is_refutes",
    "reference_is_supports",
    "nei_uncertainty_failure_proxy",
    "refutes_correction_credit_proxy",
    "supports_status_failure_proxy",
]

candidate_paths = sorted(Path("outputs").glob("**/*behavior_matrix.csv"))
if candidate_paths:
    print("Available behavior matrix candidates:")
    for path in candidate_paths:
        print(" -", path)
else:
    print("No local behavior matrix candidates found under outputs/. If you are on the server, check the run output directory.")

print("\nSelected backend:", ANALYSIS_BACKEND)
print("Meaning:", BACKEND_NOTES[ANALYSIS_BACKEND])
print("Selected input:", DATA_PATH)
if not DATA_PATH.exists():
    warnings.warn(
        f"Selected DATA_PATH does not exist: {DATA_PATH}. "
        "Pick one of the listed candidates or edit DATA_PATHS/ANALYSIS_BACKEND."
    )
print("Output:", OUTPUT_DIR.resolve())

## Load And Filter CALE Rows

The primary CFA uses only CALE variants. Direct-judge rows are excluded because they do not have the CALE construct subscores and would create structural missingness.

In [ ]:
df_raw = pd.read_csv(DATA_PATH)

if "variant" not in df_raw.columns:
    raise ValueError("Behavior matrix must include a `variant` column for CALE variant filtering.")

available_variants = sorted(df_raw["variant"].dropna().astype(str).unique().tolist())
selected_cale_variants = [variant for variant in CALE_VARIANTS if variant in available_variants]

print("Available variants in selected data:")
for variant in available_variants:
    print(" -", variant)

print("\nSelected CALE variants for pooled CFA:")
for variant in selected_cale_variants:
    print(" -", variant)

if not selected_cale_variants:
    raise ValueError(
        "None of the expected CALE variants are present. "
        f"Expected one of {CALE_VARIANTS}, found {available_variants}."
    )

missing_expected_variants = [variant for variant in CALE_VARIANTS if variant not in available_variants]
if missing_expected_variants:
    warnings.warn(
        "These CALE variants are not present in the selected data and cannot be compared here: "
        f"{missing_expected_variants}. If this is a DS strong-evaluator file, it may only include full_attack_aware_cale."
    )

missing_items = [column for column in ITEMS if column not in df_raw.columns]
if missing_items:
    raise ValueError(f"Missing CFA indicator columns: {missing_items}")

df_cale = df_raw[df_raw["variant"].isin(selected_cale_variants)].copy()
df_items = df_cale[ITEMS].apply(pd.to_numeric, errors="coerce")

missingness = df_items.isna().mean().rename("missing_share").reset_index()
missingness = missingness.rename(columns={"index": "indicator"})
missingness.to_csv(OUTPUT_DIR / "cfa_indicator_missingness.csv", index=False)

df = df_items.replace([np.inf, -np.inf], np.nan).dropna().copy()

print("Raw rows:", len(df_raw))
print("CALE rows:", len(df_cale))
print("Complete CFA rows:", len(df))
display(missingness)
display(df.head())

In [ ]:
diagnostics = []
for column in ITEMS:
    values = df[column].dropna()
    unique_values = set(values.unique().tolist())
    diagnostics.append(
        {
            "indicator": column,
            "n_unique": values.nunique(),
            "min": values.min(),
            "max": values.max(),
            "mean": values.mean(),
            "std": values.std(),
            "is_binary_0_1": unique_values.issubset({0, 1}),
            "is_bounded_0_1": values.min() >= 0 and values.max() <= 1,
        }
    )

diagnostics_df = pd.DataFrame(diagnostics)
diagnostics_df.to_csv(OUTPUT_DIR / "cfa_indicator_diagnostics.csv", index=False)
display(diagnostics_df)

if diagnostics_df["is_binary_0_1"].any():
    warnings.warn(
        "Some CFA indicators are binary. semopy treats indicators as continuous, "
        "so fit indices and standard errors should be interpreted conservatively."
    )
elif diagnostics_df["is_bounded_0_1"].any():
    warnings.warn(
        "Some CFA indicators are bounded in [0, 1]. Watch for ceiling/floor effects "
        "and interpret the continuous-indicator CFA as approximate."
    )

## CFA Model Specifications

The `three_factor` model is the primary theory-guided model. The one-factor and two-factor models are comparison baselines.

In [ ]:
FACTOR_MAP = {
    "factual_handling": ["claim_status_recognition", "correction_accuracy", "evidence_grounding", "source_faithfulness"],
    "adversarial_resistance": ["misinformation_detection", "framing_resistance", "error_rejection"],
    "boundary_control": ["hallucination_control", "uncertainty_handling"],
}

MODEL_SYNTAX = {
    "one_factor": """
cale_general =~ misinformation_detection + framing_resistance + claim_status_recognition + error_rejection + correction_accuracy + evidence_grounding + source_faithfulness + hallucination_control + uncertainty_handling
""",
    "two_factor": """
factual_handling =~ claim_status_recognition + correction_accuracy + evidence_grounding + source_faithfulness + hallucination_control + uncertainty_handling
adversarial_resistance =~ misinformation_detection + framing_resistance + error_rejection
factual_handling ~~ adversarial_resistance
""",
    "three_factor": """
factual_handling =~ claim_status_recognition + correction_accuracy + evidence_grounding + source_faithfulness
adversarial_resistance =~ misinformation_detection + framing_resistance + error_rejection
boundary_control =~ hallucination_control + uncertainty_handling
factual_handling ~~ adversarial_resistance
factual_handling ~~ boundary_control
adversarial_resistance ~~ boundary_control
""",
    "four_factor_sensitivity": """
claim_correction =~ claim_status_recognition + correction_accuracy
evidence_faithfulness =~ evidence_grounding + source_faithfulness
adversarial_resistance =~ misinformation_detection + framing_resistance + error_rejection
boundary_control =~ hallucination_control + uncertainty_handling
claim_correction ~~ evidence_faithfulness
claim_correction ~~ adversarial_resistance
claim_correction ~~ boundary_control
evidence_faithfulness ~~ adversarial_resistance
evidence_faithfulness ~~ boundary_control
adversarial_resistance ~~ boundary_control
""",
    "three_factor_nuisance_sensitivity": """
factual_handling =~ claim_status_recognition + correction_accuracy + evidence_grounding + source_faithfulness
adversarial_resistance =~ misinformation_detection + framing_resistance + error_rejection
boundary_control =~ hallucination_control + uncertainty_handling
factual_handling ~~ adversarial_resistance
factual_handling ~~ boundary_control
adversarial_resistance ~~ boundary_control
# Pre-registered residual covariances among closely related observed indicators.
evidence_grounding ~~ source_faithfulness
claim_status_recognition ~~ correction_accuracy
misinformation_detection ~~ framing_resistance
"""
}

for name, syntax in MODEL_SYNTAX.items():
    print(f"\n{name}\n{'-' * len(name)}")
    print(syntax.strip())

In [ ]:
def flatten_semopy_stats(stats_obj, model_name, n_obs):
    if isinstance(stats_obj, pd.DataFrame):
        row = stats_obj.iloc[0].to_dict() if stats_obj.shape[0] == 1 else stats_obj.stack().to_dict()
    elif isinstance(stats_obj, dict):
        row = stats_obj
    else:
        row = {"stats_repr": repr(stats_obj)}

    clean = {"model": model_name, "n_obs": n_obs}
    for key, value in row.items():
        clean_key = str(key).replace(" ", "_").replace("-", "_")
        try:
            clean[clean_key] = float(value)
        except Exception:
            clean[clean_key] = value
    return clean


def fit_cfa_model(model_name, syntax, data):
    model = Model(syntax)
    fit_result = model.fit(data)
    fit_row = flatten_semopy_stats(calc_stats(model), model_name, len(data))
    fit_row["n_indicators"] = data.shape[1]
    fit_row["fit_result"] = str(fit_result)

    estimates = model.inspect(std_est=True).copy()
    estimates.insert(0, "model", model_name)
    return model, fit_row, estimates


def extract_loadings(estimates):
    observed = set(ITEMS)
    rows = estimates[(estimates["op"] == "~") & estimates["lval"].isin(observed)].copy()
    std_col = next((c for c in ["Est. Std", "Est.Std", "Std. Estimate", "Std.Estimate"] if c in rows.columns), None)
    if std_col:
        rows = rows.rename(columns={std_col: "std_loading"})

    keep = ["model", "rval", "lval", "Estimate"]
    if "std_loading" in rows.columns:
        keep.append("std_loading")
    for optional in ["Std. Err", "z-value", "p-value"]:
        if optional in rows.columns:
            keep.append(optional)

    return rows[keep].rename(columns={"rval": "factor", "lval": "indicator"}).reset_index(drop=True)

## Fit Models

In [ ]:
fitted_models = {}
fit_rows = []
estimate_tables = []

for name, syntax in MODEL_SYNTAX.items():
    print(f"Fitting {name}...")
    try:
        model, fit_row, estimates = fit_cfa_model(name, syntax, df)
        fitted_models[name] = model
        fit_rows.append(fit_row)
        estimate_tables.append(estimates)
        print(f"  done: {name}")
    except Exception as exc:
        print(f"  failed: {name}: {exc}")
        fit_rows.append({"model": name, "n_obs": len(df), "n_indicators": df.shape[1], "error": repr(exc)})

fit_indices = pd.DataFrame(fit_rows)
parameter_estimates = pd.concat(estimate_tables, ignore_index=True) if estimate_tables else pd.DataFrame()
standardized_loadings = extract_loadings(parameter_estimates) if not parameter_estimates.empty else pd.DataFrame()

fit_indices.to_csv(OUTPUT_DIR / "cfa_fit_indices.csv", index=False)
parameter_estimates.to_csv(OUTPUT_DIR / "cfa_parameter_estimates.csv", index=False)
standardized_loadings.to_csv(OUTPUT_DIR / "cfa_standardized_loadings.csv", index=False)

with (OUTPUT_DIR / "cfa_model_syntax.json").open("w", encoding="utf-8") as file:
    json.dump(MODEL_SYNTAX, file, indent=2)

display(fit_indices)
display(standardized_loadings)

In [ ]:
preferred_fit_columns = [
    "model",
    "n_obs",
    "DoF",
    "chi2",
    "chi2_p_value",
    "chi2_p-value",
    "CFI",
    "TLI",
    "RMSEA",
    "AIC",
    "BIC",
    "LogLik",
    "error",
]

comparison = fit_indices[[c for c in preferred_fit_columns if c in fit_indices.columns]].copy()
comparison.to_csv(OUTPUT_DIR / "cfa_model_comparison.csv", index=False)
display(comparison)

## Inspect Loadings / 看每个指标是否站得住

After model fit, inspect `cfa_standardized_loadings.csv` or the displayed loadings table.

Rules of thumb / 粗略判断：

- `std_loading > .70`: strong indicator，很强。
- `.40 <= std_loading <= .70`: usable/moderate，中等，可以接受。
- `.30 <= std_loading < .40`: weak but maybe interpretable，需要理论理由。
- `< .30`: weak indicator，建议 review，不要用它支撑强 claim。

If most intended indicators load moderately/strongly onto their factors:

> Standardized loadings were generally moderate to strong, suggesting that the selected diagnostic indicators contributed meaningfully to their prespecified latent dimensions.

如果某个 indicator loading 很弱，可以写：

> The weaker loading of `[indicator]` suggests that this diagnostic signal may capture behavior not fully aligned with the prespecified factor, and should be reviewed in future instrument refinement.

特别注意：如果 `boundary_control` 只有两个 indicators，它可以跑，但解释要保守。Two-indicator factors are statistically fragile and should be treated as provisional.

In [ ]:
if not standardized_loadings.empty and "std_loading" in standardized_loadings.columns:
    loading_review = standardized_loadings.copy()
    loading_review["abs_std_loading"] = loading_review["std_loading"].abs()
    loading_review["loading_flag"] = pd.cut(
        loading_review["abs_std_loading"],
        bins=[-np.inf, 0.30, 0.40, 0.70, np.inf],
        labels=["weak_review", "borderline", "moderate", "strong"],
    )
    loading_review.to_csv(OUTPUT_DIR / "cfa_loading_review.csv", index=False)
    display(loading_review.sort_values(["model", "factor", "abs_std_loading"], ascending=[True, True, False]))
else:
    print("No standardized loading column found in semopy output. Inspect cfa_parameter_estimates.csv manually.")

## Inspect Factor Correlations / 看 latent dimensions 是否可区分

Factor correlations answer whether dimensions are related but separable.

Interpretation / 解释：

- `.00-.30`: weak relation，可能是比较独立的 dimensions。
- `.30-.80`: useful range，related but distinguishable，通常最适合你的 validity story。
- `.80-.90`: very high，需要小心，可能有 strong general factor。
- `>.90`: factors may not be empirically distinguishable，不能强说是清楚分开的 latent traits。

Possible claim if correlations are moderate:

> The latent factors were positively correlated but not redundant, supporting the interpretation of CALE behavior as a set of related but distinguishable diagnostic dimensions.

Possible claim if correlations are too high:

> The high factor correlations suggest that the current indicators may be dominated by a general CALE behavior factor, and that the proposed subdimensions require further evidence or item refinement.

In [ ]:
if not parameter_estimates.empty:
    observed = set(ITEMS)
    factor_covariances = parameter_estimates[
        (parameter_estimates["op"] == "~~")
        & (~parameter_estimates["lval"].isin(observed))
        & (~parameter_estimates["rval"].isin(observed))
        & (parameter_estimates["lval"] != parameter_estimates["rval"])
    ].copy()
    std_col = next((c for c in ["Est. Std", "Est.Std", "Std. Estimate", "Std.Estimate"] if c in factor_covariances.columns), None)
    if std_col:
        factor_covariances = factor_covariances.rename(columns={std_col: "std_factor_correlation"})
    factor_covariances.to_csv(OUTPUT_DIR / "cfa_factor_correlations.csv", index=False)
    display(factor_covariances)
else:
    print("No parameter estimates available.")

## How To Interpret Model Comparison / 怎么判断模型比较

Use the table above as the first decision point.

### If `three_factor` is best or clearly better than `one_factor`

Possible claim / 可以写：

> The theory-guided three-factor CFA-style model fit the CALE behavior matrix better than a one-factor baseline, suggesting that CALE diagnostic indicators are better represented as multiple related dimensions than as a single undifferentiated quality score.

中文解释：这支持你的核心故事：CALE 的价值不是只产出一个 final score，而是把 evaluator behavior 拆成多个可诊断、可建模的 measurement dimensions。

### If `one_factor` fits similarly to `three_factor`

Possible claim / 可以写：

> The CFA-style model comparison did not clearly favor the multidimensional specification over a simpler one-factor model. This suggests that the current indicators may share a strong general CALE behavior factor, while finer-grained dimensions require further evidence.

中文解释：不能硬说三维结构被确认了。可以改讲 general CALE behavior factor 很强，subdimensions 还是 exploratory。

### If all models fit poorly

Possible claim / 可以写：

> The current CFA-style results provide limited support for the prespecified measurement model, indicating that the diagnostic indicators may need refinement or that the bounded/binary scoring procedure is not well captured by continuous-indicator CFA.

中文解释：这不是失败，反而是 measurement paper 里很合理的发现：当前 checklist indicators 可能需要修订，或者需要 ordinal CFA / WLSMV。

## Target-Specific Robustness

This repeats the model comparison for pooled, Qwen-only, and Llama-only rows. Treat these as robustness checks, not as separate fully validated models.

In [ ]:
def fit_subset(label, mask):
    subset_items = df_cale.loc[mask, ITEMS].apply(pd.to_numeric, errors="coerce")
    subset = subset_items.replace([np.inf, -np.inf], np.nan).dropna()
    rows = []
    for name, syntax in MODEL_SYNTAX.items():
        try:
            _, fit_row, _ = fit_cfa_model(name, syntax, subset)
            fit_row["subset"] = label
            rows.append(fit_row)
        except Exception as exc:
            rows.append({"subset": label, "model": name, "n_obs": len(subset), "error": repr(exc)})
    return pd.DataFrame(rows)


robustness_tables = []
all_rows = pd.Series(True, index=df_cale.index)
robustness_tables.append(fit_subset("pooled", all_rows))

if "model_name" in df_cale.columns:
    target = df_cale["model_name"].astype(str).str.lower()
    robustness_tables.append(fit_subset("qwen_only", target.str.contains("qwen", na=False)))
    robustness_tables.append(fit_subset("llama_only", target.str.contains("llama", na=False)))

if "variant" in df_cale.columns:
    for variant in selected_cale_variants:
        robustness_tables.append(fit_subset(f"variant_{variant}", df_cale["variant"].eq(variant)))

robustness = pd.concat(robustness_tables, ignore_index=True)
robustness.to_csv(OUTPUT_DIR / "cfa_robustness_fit_indices.csv", index=False)
display(robustness[[c for c in ["subset", *preferred_fit_columns] if c in robustness.columns]])

## Robustness Interpretation / 稳健性怎么看

Use `cfa_robustness_fit_indices.csv` to ask whether the same measurement story holds across subsets.

### If Qwen-only and Llama-only show similar model rankings

Possible claim / 可以写：

> The same model ranking was observed across target-model subsets, suggesting that the internal structure was not driven by a single target model's response distribution.

中文解释：这说明 CFA 结构不是 Qwen 或 Llama 某一个模型独有的产物。

### If variant-specific subsets show similar model rankings

Possible claim / 可以写：

> The model structure was broadly consistent across CALE variants, suggesting that the proposed dimensions reflect stable diagnostic organization rather than a variant-specific artifact.

### If robustness differs strongly across subsets

Possible claim / 可以写：

> The CFA-style structure varied across subsets, indicating that CALE's internal diagnostic organization may depend on target model, evaluator variant, or item distribution. This motivates follow-up measurement-invariance analyses.

中文解释：这时候不要说 universal structure。可以说结构可能 condition-dependent，这也很有论文价值。

## CALE Family Psychometrics Runner / CALE 家族心理测量实验面板

This section is the main experimental workflow. It keeps each case and group separate, then saves comparable fit, loading, factor-correlation, convergent-validity, and discriminant-validity tables.

这一节是主实验工作流：每个 case 和 group 分开跑，同时输出可横向比较的 fit、loading、factor correlation、convergent validity 和 discriminant validity 表。

Default behavior / 默认行为：

- primary case: `heuristic_full`, because it is the only full-sample CALE-family run.
- comparison cases: `qwen25_7b_limit1000` and `deepseek_v4_pro_limit1000`, when available, for backend robustness.
- subset separation: pooled CALE rows, each CALE variant, each target model, and each reference label.

What this answers / 它能回答：

- CALE family 是否表现出 multidimensional internal structure？
- `generic_cale` / `attack_aware_cale` / `full_attack_aware_cale` 的 measurement structure 是否相似？
- Qwen target vs Llama target 是否结构相似？
- strong evaluator backend 是否保留类似的 `full_attack_aware_cale` structure？

Important / 注意：如果某个 strong evaluator file 只包含 `full_attack_aware_cale`，它只能做 backend robustness，不能回答完整 CALE family 的 variant comparison。


In [ ]:
import matplotlib.pyplot as plt

ONE_CLICK_OUTPUT_DIR = Path("figures/cfa_behavior_model/family_psychometrics")
ONE_CLICK_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ANALYSIS_CASES = {
    "ds_limit1000": {
        "path": Path("outputs/small_models_all/fever_dev_qwen25_15b_llama32_1b_neutral_deepseek_v4_pro_limit1000_eval_behavior_matrix.csv"),
        "role": "backend_robustness",
        "claim_scope": "DeepSeek evaluator backend robustness for full_attack_aware_cale on the strong-evaluator subset.",
    },
    "ds_limit20_smoke": {
        "path": Path("outputs/small_models_all/fever_dev_qwen25_15b_llama32_1b_neutral_deepseek_v4_pro_limit20_eval_behavior_matrix.csv"),
        "role": "smoke_test_only",
        "claim_scope": "Pipeline sanity check only; too small for substantive CFA claims.",
    },
    "qwen25_7b_limit1000": {
        "path": Path("outputs/small_models_all/fever_dev_qwen25_15b_llama32_1b_neutral_strong_qwen25_7b_limit1000_eval_behavior_matrix.csv"),
        "role": "backend_robustness",
        "claim_scope": "Qwen2.5-7B evaluator backend robustness for full_attack_aware_cale on the strong-evaluator subset.",
    },
    "heuristic_full": {
        "path": Path("outputs/small_models_all/fever_dev_qwen25_15b_llama32_1b_neutral_full_eval_behavior_matrix.csv"),
        "role": "primary_family_psychometrics",
        "claim_scope": "Primary full-sample CALE-family psychometrics evidence.",
    },
}

# Default: run the primary full-sample CALE-family case. Add strong evaluator cases when available.
RUN_CASE_KEYS = ["heuristic_full"]
# Backend robustness comparison, when files are available:
# RUN_CASE_KEYS = ["heuristic_full", "qwen25_7b_limit1000", "ds_limit1000"]

MIN_N_FOR_CFA = 30
MAX_REFERENCE_LABEL_SUBSETS = 3

def safe_name(value):
    return str(value).replace("/", "_").replace(" ", "_").replace(":", "_")


def present_cale_variants(data):
    if "variant" not in data.columns:
        return []
    available = set(data["variant"].dropna().astype(str))
    return [variant for variant in CALE_VARIANTS if variant in available]


def build_experimental_subsets(data):
    subsets = []
    variants = present_cale_variants(data)
    if variants:
        subsets.append({"subset": "pooled_cale", "mask": data["variant"].isin(variants), "basis": "all available CALE variants"})
        for variant in variants:
            subsets.append({"subset": f"variant_{variant}", "mask": data["variant"].eq(variant), "basis": f"variant == {variant}"})

    if "model_name" in data.columns and variants:
        model_name = data["model_name"].astype(str).str.lower()
        subsets.append({"subset": "target_qwen", "mask": data["variant"].isin(variants) & model_name.str.contains("qwen", na=False), "basis": "CALE rows, qwen target only"})
        subsets.append({"subset": "target_llama", "mask": data["variant"].isin(variants) & model_name.str.contains("llama", na=False), "basis": "CALE rows, llama target only"})

    if "reference_label" in data.columns and variants:
        labels = data["reference_label"].dropna().astype(str).value_counts().head(MAX_REFERENCE_LABEL_SUBSETS).index.tolist()
        for label in labels:
            subsets.append({"subset": f"reference_{safe_name(label)}", "mask": data["variant"].isin(variants) & data["reference_label"].astype(str).eq(label), "basis": f"CALE rows, reference_label == {label}"})

    return subsets


def prepare_items_for_cfa(data, mask):
    selected = data.loc[mask].copy()
    if selected.empty:
        return selected, pd.DataFrame(), pd.DataFrame()
    missing = [column for column in ITEMS if column not in selected.columns]
    if missing:
        raise ValueError(f"Missing CFA item columns: {missing}")
    item_data = selected[ITEMS].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
    complete = item_data.dropna().copy()
    varying = complete.nunique(dropna=True)
    kept_columns = varying[varying > 1].index.tolist()
    complete = complete[kept_columns]
    return selected, item_data, complete


def subset_model_syntax(item_columns):
    item_columns = set(item_columns)
    specs = {}
    for model_name, syntax in MODEL_SYNTAX.items():
        if all(item in item_columns for item in ITEMS):
            specs[model_name] = syntax
        else:
            # If a subset has zero variance on some indicators, skip CFA rather than silently changing the model.
            specs[model_name] = None
    return specs


def extract_factor_correlations(estimates):
    if estimates.empty:
        return pd.DataFrame()
    observed = set(ITEMS)
    rows = estimates[
        (estimates["op"] == "~~")
        & (~estimates["lval"].isin(observed))
        & (~estimates["rval"].isin(observed))
        & (estimates["lval"] != estimates["rval"])
    ].copy()
    std_col = next((c for c in ["Est. Std", "Est.Std", "Std. Estimate", "Std.Estimate"] if c in rows.columns), None)
    if std_col:
        rows = rows.rename(columns={std_col: "std_factor_correlation"})
    return rows


def run_case_subset(case_key, case_info, subset_info, data):
    subset_name = subset_info["subset"]
    case_dir = ONE_CLICK_OUTPUT_DIR / case_key / safe_name(subset_name)
    case_dir.mkdir(parents=True, exist_ok=True)

    selected_rows, item_data, complete = prepare_items_for_cfa(data, subset_info["mask"])
    missingness = item_data.isna().mean().rename("missing_share").reset_index().rename(columns={"index": "indicator"})
    missingness.to_csv(case_dir / "indicator_missingness.csv", index=False)
    selected_rows.to_csv(case_dir / "behavior_matrix_subset.csv", index=False)

    fit_rows = []
    estimates_all = []
    loadings_all = []
    corrs_all = []

    if len(complete) < MIN_N_FOR_CFA:
        fit_rows.append({
            "case": case_key,
            "subset": subset_name,
            "model": "all_models",
            "n_obs": len(complete),
            "status": "skipped_low_n",
            "error": f"Need at least {MIN_N_FOR_CFA} complete rows for CFA.",
        })
    elif set(complete.columns) != set(ITEMS):
        fit_rows.append({
            "case": case_key,
            "subset": subset_name,
            "model": "all_models",
            "n_obs": len(complete),
            "status": "skipped_zero_variance_items",
            "error": f"Some CFA items have zero variance in this subset: {sorted(set(ITEMS) - set(complete.columns))}",
        })
    else:
        for model_name, syntax in MODEL_SYNTAX.items():
            try:
                _, fit_row, estimates = fit_cfa_model(model_name, syntax, complete)
                fit_row.update({
                    "case": case_key,
                    "subset": subset_name,
                    "case_role": case_info["role"],
                    "claim_scope": case_info["claim_scope"],
                    "subset_basis": subset_info["basis"],
                    "status": "fit_ok",
                    "data_path": str(case_info["path"]),
                })
                fit_rows.append(fit_row)

                # Keep raw semopy estimates for extraction. Add case/subset only
                # to copied output tables; otherwise repeated insertions can create
                # false fit_failed rows such as "cannot insert case, already exists".
                estimates_annotated = estimates.copy()
                estimates_annotated.insert(1, "case", case_key)
                estimates_annotated.insert(2, "subset", subset_name)
                estimates_all.append(estimates_annotated)

                loadings = extract_loadings(estimates)
                loadings.insert(0, "case", case_key)
                loadings.insert(1, "subset", subset_name)
                loadings_all.append(loadings)

                corrs = extract_factor_correlations(estimates)
                if not corrs.empty:
                    corrs.insert(0, "case", case_key)
                    corrs.insert(1, "subset", subset_name)
                    corrs_all.append(corrs)
            except Exception as exc:
                fit_rows.append({
                    "case": case_key,
                    "subset": subset_name,
                    "model": model_name,
                    "n_obs": len(complete),
                    "case_role": case_info["role"],
                    "claim_scope": case_info["claim_scope"],
                    "subset_basis": subset_info["basis"],
                    "status": "fit_failed",
                    "error": repr(exc),
                    "data_path": str(case_info["path"]),
                })

    fit_df = pd.DataFrame(fit_rows)
    est_df = pd.concat(estimates_all, ignore_index=True) if estimates_all else pd.DataFrame()
    load_df = pd.concat(loadings_all, ignore_index=True) if loadings_all else pd.DataFrame()
    corr_df = pd.concat(corrs_all, ignore_index=True) if corrs_all else pd.DataFrame()

    fit_df.to_csv(case_dir / "fit_indices.csv", index=False)
    est_df.to_csv(case_dir / "parameter_estimates.csv", index=False)
    load_df.to_csv(case_dir / "standardized_loadings.csv", index=False)
    corr_df.to_csv(case_dir / "factor_correlations.csv", index=False)
    return fit_df, load_df, corr_df


def run_experimental_cfa(case_keys):
    all_fit = []
    all_loadings = []
    all_corrs = []
    manifest = []

    for case_key in case_keys:
        case_info = ANALYSIS_CASES[case_key]
        path = case_info["path"]
        if not path.exists():
            manifest.append({"case": case_key, "path": str(path), "status": "missing_file", "claim_scope": case_info["claim_scope"]})
            print(f"Skipping missing case {case_key}: {path}")
            continue

        data = pd.read_csv(path)
        variants = present_cale_variants(data)
        subsets = build_experimental_subsets(data)
        manifest.append({
            "case": case_key,
            "path": str(path),
            "status": "loaded",
            "rows": len(data),
            "available_cale_variants": ", ".join(variants),
            "n_subsets": len(subsets),
            "claim_scope": case_info["claim_scope"],
        })
        print(f"\nCase {case_key}: {len(data)} rows, variants={variants}")

        for subset_info in subsets:
            print(f"  subset {subset_info['subset']} ({subset_info['basis']})")
            fit_df, load_df, corr_df = run_case_subset(case_key, case_info, subset_info, data)
            all_fit.append(fit_df)
            if not load_df.empty:
                all_loadings.append(load_df)
            if not corr_df.empty:
                all_corrs.append(corr_df)

    manifest_df = pd.DataFrame(manifest)
    fit_all = pd.concat(all_fit, ignore_index=True) if all_fit else pd.DataFrame()
    load_all = pd.concat(all_loadings, ignore_index=True) if all_loadings else pd.DataFrame()
    corr_all = pd.concat(all_corrs, ignore_index=True) if all_corrs else pd.DataFrame()

    manifest_df.to_csv(ONE_CLICK_OUTPUT_DIR / "analysis_manifest.csv", index=False)
    fit_all.to_csv(ONE_CLICK_OUTPUT_DIR / "all_cases_fit_indices.csv", index=False)
    load_all.to_csv(ONE_CLICK_OUTPUT_DIR / "all_cases_standardized_loadings.csv", index=False)
    corr_all.to_csv(ONE_CLICK_OUTPUT_DIR / "all_cases_factor_correlations.csv", index=False)
    return manifest_df, fit_all, load_all, corr_all

In [ ]:
manifest_df, fit_all, load_all, corr_all = run_experimental_cfa(RUN_CASE_KEYS)

print("\nManifest")
display(manifest_df)

print("\nFit summary")
display(fit_all[[c for c in ["case", "subset", "model", "n_obs", "status", "DoF", "chi2", "CFI", "TLI", "RMSEA", "AIC", "BIC", "error"] if c in fit_all.columns]])

print("\nLoadings summary")
display(load_all.head(30) if not load_all.empty else load_all)

print("\nFactor correlations summary")
display(corr_all.head(30) if not corr_all.empty else corr_all)

## Convergent And Discriminant Validity Checks / 收敛与区分效度检查

These checks operationalize the validity story after CFA-style models have been fit.

这些检查把 validity story 量化成几张表。

**Convergent validity** means construct-relevant signals should move together:

- indicators assigned to the same factor should have moderate/strong standardized loadings;
- average variance extracted (AVE) and composite reliability (CR) should be non-trivial;
- construct indicators should correlate with relevant criteria such as `final_score` and reference-label-specific proxies.

**Discriminant validity** means different constructs should be related but not redundant, and should not mainly track nuisance signals:

- factor correlations should not approach 1;
- Fornell-Larcker-style checks should show each factor's AVE is not smaller than squared factor correlations;
- raw indicators should not be more strongly associated with nuisance variables (`target_model`, `reference_label`, `domain`, `risk_level`, `framing_style`) than with construct-relevant signals.

These are screening checks, not final psychometric proof.

这些是 screening checks，不是最终心理测量证明。


In [ ]:
VALIDITY_OUTPUT_DIR = Path("figures/cfa_behavior_model/family_psychometrics/validity")
VALIDITY_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CONSTRUCT_RELEVANT_SIGNALS = [
    "final_score",
    "uncertainty",
    "nei_uncertainty_failure_proxy",
    "refutes_correction_credit_proxy",
    "supports_status_failure_proxy",
]

CONSTRUCT_IRRELEVANT_NUISANCE_SIGNALS = [
    "model_name",
    "reference_label",
    "dataset",
    "domain",
    "risk_level",
    "framing_style",
]


def reliability_from_loadings(loadings):
    if loadings.empty or "std_loading" not in loadings.columns:
        return pd.DataFrame()
    data = loadings[loadings["model"].eq("three_factor")].copy()
    rows = []
    for keys, group in data.groupby(["case", "subset", "factor"], dropna=False):
        case, subset, factor = keys
        lambdas = pd.to_numeric(group["std_loading"], errors="coerce").dropna()
        if lambdas.empty:
            continue
        error_var = 1 - np.square(lambdas)
        composite_reliability = (lambdas.sum() ** 2) / ((lambdas.sum() ** 2) + error_var.sum()) if ((lambdas.sum() ** 2) + error_var.sum()) else np.nan
        rows.append({
            "case": case,
            "subset": subset,
            "factor": factor,
            "n_indicators": len(lambdas),
            "mean_abs_loading": lambdas.abs().mean(),
            "ave": np.square(lambdas).mean(),
            "composite_reliability": composite_reliability,
        })
    return pd.DataFrame(rows)


def fornell_larcker_from_outputs(reliability, corrs):
    if reliability.empty or corrs.empty or "std_factor_correlation" not in corrs.columns:
        return pd.DataFrame()
    rel = reliability[["case", "subset", "factor", "ave"]].copy()
    rows = []
    corr_data = corrs[corrs["model"].eq("three_factor")].copy() if "model" in corrs.columns else corrs.copy()
    for _, row in corr_data.iterrows():
        case = row.get("case")
        subset = row.get("subset")
        left = row.get("lval")
        right = row.get("rval")
        corr = pd.to_numeric(pd.Series([row.get("std_factor_correlation")]), errors="coerce").iloc[0]
        left_ave = rel[(rel["case"].eq(case)) & (rel["subset"].eq(subset)) & (rel["factor"].eq(left))]["ave"]
        right_ave = rel[(rel["case"].eq(case)) & (rel["subset"].eq(subset)) & (rel["factor"].eq(right))]["ave"]
        if left_ave.empty or right_ave.empty or pd.isna(corr):
            continue
        rows.append({
            "case": case,
            "subset": subset,
            "factor_left": left,
            "factor_right": right,
            "factor_correlation": corr,
            "squared_factor_correlation": corr ** 2,
            "left_ave": left_ave.iloc[0],
            "right_ave": right_ave.iloc[0],
            "fornell_larcker_pass": min(left_ave.iloc[0], right_ave.iloc[0]) > corr ** 2,
        })
    return pd.DataFrame(rows)


def correlation_screen_for_case(case_key, case_info):
    path = case_info["path"]
    if not path.exists():
        return pd.DataFrame()
    data = pd.read_csv(path)
    variants = present_cale_variants(data)
    if not variants:
        return pd.DataFrame()
    data = data[data["variant"].isin(variants)].copy()
    rows = []
    signal_columns = [c for c in CONSTRUCT_RELEVANT_SIGNALS if c in data.columns]
    nuisance_columns = [c for c in CONSTRUCT_IRRELEVANT_NUISANCE_SIGNALS if c in data.columns]
    for signal in signal_columns:
        data[f"__signal__{signal}"] = pd.to_numeric(data[signal], errors="coerce")
    for signal in nuisance_columns:
        data[f"__nuisance__{signal}"] = pd.factorize(data[signal].astype(str), sort=True)[0]
    encoded_columns = [f"__signal__{c}" for c in signal_columns] + [f"__nuisance__{c}" for c in nuisance_columns]
    for indicator in ITEMS:
        if indicator not in data.columns:
            continue
        x = pd.to_numeric(data[indicator], errors="coerce")
        for encoded in encoded_columns:
            y = pd.to_numeric(data[encoded], errors="coerce")
            valid = x.notna() & y.notna()
            if valid.sum() < 30 or x[valid].nunique() <= 1 or y[valid].nunique() <= 1:
                continue
            rows.append({
                "case": case_key,
                "indicator": indicator,
                "signal": encoded.replace("__signal__", "").replace("__nuisance__", ""),
                "signal_type": "construct_relevant" if encoded.startswith("__signal__") else "construct_irrelevant_nuisance",
                "n": int(valid.sum()),
                "pearson_abs": abs(x[valid].corr(y[valid], method="pearson")),
                "spearman_abs": abs(x[valid].corr(y[valid], method="spearman")),
            })
    return pd.DataFrame(rows)

reliability_summary = reliability_from_loadings(load_all if "load_all" in globals() else pd.DataFrame())
fornell_larcker = fornell_larcker_from_outputs(reliability_summary, corr_all if "corr_all" in globals() else pd.DataFrame())
validity_correlation_screens = []
for case_key in RUN_CASE_KEYS:
    if case_key in ANALYSIS_CASES:
        validity_correlation_screens.append(correlation_screen_for_case(case_key, ANALYSIS_CASES[case_key]))
validity_correlations = pd.concat(validity_correlation_screens, ignore_index=True) if validity_correlation_screens else pd.DataFrame()

reliability_summary.to_csv(VALIDITY_OUTPUT_DIR / "cfa_family_convergent_reliability_ave_cr.csv", index=False)
fornell_larcker.to_csv(VALIDITY_OUTPUT_DIR / "cfa_family_discriminant_validity_fornell_larcker.csv", index=False)
validity_correlations.to_csv(VALIDITY_OUTPUT_DIR / "cfa_family_indicator_signal_correlations.csv", index=False)

if not validity_correlations.empty:
    nuisance_vs_relevant = validity_correlations.groupby(["case", "signal_type"], dropna=False)[["pearson_abs", "spearman_abs"]].mean().reset_index()
    nuisance_vs_relevant.to_csv(VALIDITY_OUTPUT_DIR / "cfa_family_nuisance_vs_construct_relevant_summary.csv", index=False)
else:
    nuisance_vs_relevant = pd.DataFrame()

display(reliability_summary.head(30))
display(fornell_larcker.head(30))
display(nuisance_vs_relevant)


## Psychometric Claim Readiness / 心理测量结论成熟度表

This section turns the CFA-style outputs into a thesis-facing claim-readiness table. It is intentionally conservative: the current `semopy` models treat bounded/binary CALE indicators as continuous, so the table describes preliminary screening evidence rather than definitive psychometric validation.

这一节把 CFA-style 输出整理成可以写进论文的 claim-readiness table。它的口径故意保守：当前 `semopy` 模型把 bounded/binary CALE indicators 当作连续变量处理，所以这里只能说 preliminary screening evidence，不能说已经完成严格 validation。



In [ ]:
CLAIM_READINESS_OUTPUT = VALIDITY_OUTPUT_DIR / "cfa_claim_readiness.csv"


def _has_rows(name):
    value = globals().get(name, pd.DataFrame())
    return isinstance(value, pd.DataFrame) and not value.empty


def _count_fit_ok(model_name=None):
    fit = globals().get("fit_all", pd.DataFrame())
    if not isinstance(fit, pd.DataFrame) or fit.empty or "status" not in fit.columns:
        return 0
    data = fit[fit["status"].eq("fit_ok")]
    if model_name and "model" in data.columns:
        data = data[data["model"].eq(model_name)]
    return len(data)


def _subsets_with_fit():
    fit = globals().get("fit_all", pd.DataFrame())
    if not isinstance(fit, pd.DataFrame) or fit.empty or "status" not in fit.columns:
        return []
    ok = fit[fit["status"].eq("fit_ok")].copy()
    if "subset" not in ok.columns:
        return []
    return sorted(ok["subset"].dropna().astype(str).unique().tolist())


def _mean_value(frame_name, column):
    frame = globals().get(frame_name, pd.DataFrame())
    if not isinstance(frame, pd.DataFrame) or frame.empty or column not in frame.columns:
        return None
    values = pd.to_numeric(frame[column], errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()
    if values.empty:
        return None
    return float(values.mean())


def _validity_signal_summary():
    data = globals().get("validity_correlations", pd.DataFrame())
    if not isinstance(data, pd.DataFrame) or data.empty or "signal_type" not in data.columns:
        return "No indicator-signal correlation screen has been generated yet."
    parts = []
    for signal_type, group in data.groupby("signal_type", dropna=False):
        pearson = pd.to_numeric(group.get("pearson_abs"), errors="coerce").mean()
        spearman = pd.to_numeric(group.get("spearman_abs"), errors="coerce").mean()
        parts.append(f"{signal_type}: mean |Pearson|={pearson:.3f}, mean |Spearman|={spearman:.3f}, pairs={len(group)}")
    return "; ".join(parts)


def _fit_evidence_summary():
    three_factor_ok = _count_fit_ok("three_factor")
    all_fit_ok = _count_fit_ok()
    subsets = _subsets_with_fit()
    if all_fit_ok == 0:
        return "No successful CFA-style model fits are currently available in this notebook state."
    subset_note = ", ".join(subsets[:8]) + (" ..." if len(subsets) > 8 else "")
    return f"{all_fit_ok} successful CFA-style fits, including {three_factor_ok} three-factor fits; fitted subsets: {subset_note}."


def _loading_evidence_summary():
    loadings = globals().get("load_all", pd.DataFrame())
    if not isinstance(loadings, pd.DataFrame) or loadings.empty or "std_loading" not in loadings.columns:
        return "No standardized loading table is currently available."
    data = loadings[loadings.get("model", "").eq("three_factor")].copy() if "model" in loadings.columns else loadings.copy()
    values = pd.to_numeric(data["std_loading"], errors="coerce").abs().dropna()
    if values.empty:
        return "Standardized loading table exists, but no numeric standardized loadings were found."
    return f"Three-factor standardized loadings: mean |loading|={values.mean():.3f}, median |loading|={values.median():.3f}, n={len(values)}."


def _reliability_evidence_summary():
    reliability = globals().get("reliability_summary", pd.DataFrame())
    if not isinstance(reliability, pd.DataFrame) or reliability.empty:
        return "No AVE/CR reliability table has been generated yet."
    ave = pd.to_numeric(reliability.get("ave"), errors="coerce").mean()
    cr = pd.to_numeric(reliability.get("composite_reliability"), errors="coerce").mean()
    return f"AVE/CR screening table generated; mean AVE={ave:.3f}, mean CR={cr:.3f}, rows={len(reliability)}."


def _discriminant_evidence_summary():
    parts = []
    fornell = globals().get("fornell_larcker", pd.DataFrame())
    if isinstance(fornell, pd.DataFrame) and not fornell.empty:
        if "fornell_larcker_pass" in fornell.columns:
            pass_rate = pd.to_numeric(fornell["fornell_larcker_pass"], errors="coerce").mean()
            parts.append(f"Fornell-Larcker screening generated; pass share={pass_rate:.3f}, rows={len(fornell)}")
        else:
            parts.append(f"Fornell-Larcker screening generated; rows={len(fornell)}")
    corrs = globals().get("corr_all", pd.DataFrame())
    if isinstance(corrs, pd.DataFrame) and not corrs.empty and "std_factor_correlation" in corrs.columns:
        values = pd.to_numeric(corrs["std_factor_correlation"], errors="coerce").replace([np.inf, -np.inf], np.nan)
        finite = values.dropna()
        admissible = finite[finite.abs().le(1)]
        inadmissible_count = int(finite.abs().gt(1).sum() + values.isna().sum())
        if admissible.empty:
            parts.append(f"factor-correlation table generated; no admissible standardized correlations in [-1, 1], inadmissible/missing rows={inadmissible_count}/{len(corrs)}")
        else:
            parts.append(f"factor-correlation table generated; admissible mean |corr|={admissible.abs().mean():.3f}, admissible rows={len(admissible)}/{len(corrs)}, inadmissible/missing rows={inadmissible_count}")
    if _has_rows("nuisance_vs_relevant"):
        parts.append("nuisance-vs-construct-relevant indicator correlation summary generated")
    return "; ".join(parts) if parts else "No discriminant-validity screening tables are currently available."


def _invariance_evidence_summary():
    subsets = _subsets_with_fit()
    target_subsets = [s for s in subsets if "target_qwen" in s or "target_llama" in s]
    if target_subsets:
        return f"Target-specific subset fits exist ({', '.join(target_subsets)}), which can support a robustness screen. Formal configural/metric/scalar invariance has not been fit."
    return "No formal measurement-invariance model has been fit; target/backend subset CFA can only be interpreted as a robustness screen if available."


def build_claim_readiness_table():
    rows = [
        {
            "claim_area": "internal_structure",
            "status": "supported_preliminarily" if _count_fit_ok("three_factor") > 0 and _has_rows("load_all") else "not_evaluated_yet",
            "evidence": _fit_evidence_summary() + " " + _loading_evidence_summary(),
            "limitations": "CFA-style semopy models approximate bounded/binary CALE indicators as continuous; model fit does not prove a definitive ordinal psychometric structure.",
            "safe_wording": "The CALE indicators show preliminary internal-structure evidence consistent with interpretable latent dimensions, pending stricter ordinal and invariance checks.",
            "supporting_outputs": "all_cases_fit_indices.csv; all_cases_standardized_loadings.csv; all_cases_factor_correlations.csv",
            "recommended_next_step": "Compare one-factor vs correlated-factor models on the same balanced subsets and report loadings/factor correlations with conservative language.",
        },
        {
            "claim_area": "convergent_validity",
            "status": "screening_evidence_only" if _has_rows("reliability_summary") or _has_rows("validity_correlations") else "not_evaluated_yet",
            "evidence": _reliability_evidence_summary() + " " + _validity_signal_summary(),
            "limitations": "AVE/CR and raw indicator-signal correlations are screening evidence; they are not external criterion validation and do not yet use latent factor scores.",
            "safe_wording": "Construct-relevant indicators show preliminary convergence with related scores/proxies, but this should be described as a validity screen rather than confirmed convergent validity.",
            "supporting_outputs": "validity/cfa_family_convergent_reliability_ave_cr.csv; validity/cfa_family_indicator_signal_correlations.csv",
            "recommended_next_step": "Add latent factor or construct-scale scores and correlate them with theoretically relevant external/proxy criteria on matched subsets.",
        },
        {
            "claim_area": "discriminant_validity",
            "status": "needs_stronger_nuisance_tests" if _has_rows("fornell_larcker") or _has_rows("nuisance_vs_relevant") or _has_rows("corr_all") else "not_evaluated_yet",
            "evidence": _discriminant_evidence_summary(),
            "limitations": "Current nuisance checks encode categorical variables with simple factor codes; this is useful for screening but weaker than eta-squared, permutation, or grouped invariance analyses.",
            "safe_wording": "The current results provide preliminary discriminant-validity screening; stronger nuisance-sensitivity tests are needed before claiming constructs are clearly separable from irrelevant grouping factors.",
            "supporting_outputs": "validity/cfa_family_discriminant_validity_fornell_larcker.csv; validity/cfa_family_nuisance_vs_construct_relevant_summary.csv",
            "recommended_next_step": "Replace factor-code correlations for categorical nuisance variables with eta-squared/permutation tests and compare factor correlations against AVE.",
        },
        {
            "claim_area": "measurement_invariance",
            "status": "not_fully_tested",
            "evidence": _invariance_evidence_summary(),
            "limitations": "Target/backend/variant subset fits are not the same as formal configural, metric, or scalar invariance tests.",
            "safe_wording": "Subset analyses can be reported as robustness checks, not as evidence of established measurement invariance.",
            "supporting_outputs": "all_cases_fit_indices.csv; all_cases_standardized_loadings.csv; target-specific subset folders when generated",
            "recommended_next_step": "Add a loading-similarity or multi-group invariance screen across Qwen/Llama target splits and evaluator backends.",
        },
        {
            "claim_area": "compact_capability_score",
            "status": "exploratory",
            "evidence": "The CFA/PCA structure can motivate lower-dimensional summaries, but this notebook does not yet validate factor scores as target-model capability scores.",
            "limitations": "Compact scores may mix target-model behavior, evaluator backend behavior, and protocol effects unless backend/variant are fixed and matched subsets are used.",
            "safe_wording": "CALE-derived compact dimensions can be used as exploratory diagnostic summaries of target responses under a specified evaluator backend/protocol, not as calibrated universal capability scores.",
            "supporting_outputs": "all_cases_standardized_loadings.csv; downstream compact summaries should be produced in the global evaluator audit notebook",
            "recommended_next_step": "Create target-model capability summaries only after fixing evaluator backend and evaluator variant, then test robustness across matched Qwen/Llama subsets.",
        },
    ]
    return pd.DataFrame(rows)


claim_readiness = build_claim_readiness_table()
claim_readiness.to_csv(CLAIM_READINESS_OUTPUT, index=False)
print("Saved psychometric claim-readiness table to:", CLAIM_READINESS_OUTPUT.resolve())
display(claim_readiness)



## One-Click Visualizations / 一键可视化

These plots are deliberately simple and file-based, so they work on server notebooks without extra visualization libraries.

这些图尽量只用 matplotlib，方便服务器环境直接跑。

In [ ]:
def save_metric_barplot(data, metric, path, title):
    if data.empty or metric not in data.columns:
        return
    if "status" in data.columns:
        plot_data = data[data["status"].eq("fit_ok")].copy()
    else:
        plot_data = data.copy()
    plot_data = plot_data.dropna(subset=[metric])
    if plot_data.empty:
        return
    plot_data["label"] = plot_data["case"].astype(str) + " | " + plot_data["subset"].astype(str) + " | " + plot_data["model"].astype(str)
    plot_data = plot_data.sort_values(metric)
    fig, ax = plt.subplots(figsize=(11, max(4, 0.32 * len(plot_data))))
    ax.barh(plot_data["label"], plot_data[metric], color="#4C78A8")
    ax.set_title(title)
    ax.set_xlabel(metric)
    ax.tick_params(axis="y", labelsize=7)
    fig.tight_layout()
    fig.savefig(path, dpi=220)
    plt.close(fig)


def save_loading_heatmap_table(loadings, model_name, path):
    if loadings.empty or "std_loading" not in loadings.columns:
        return
    data = loadings[loadings["model"].eq(model_name)].copy()
    if data.empty:
        return
    data["case_subset"] = data["case"].astype(str) + " | " + data["subset"].astype(str)
    table = data.pivot_table(index="case_subset", columns="indicator", values="std_loading", aggfunc="mean")
    if table.empty:
        return
    fig, ax = plt.subplots(figsize=(max(9, 0.75 * len(table.columns)), max(4, 0.42 * len(table))))
    image = ax.imshow(table.values, vmin=-1, vmax=1, cmap="RdBu_r", aspect="auto")
    ax.set_xticks(range(len(table.columns)))
    ax.set_xticklabels(table.columns, rotation=35, ha="right", fontsize=8)
    ax.set_yticks(range(len(table.index)))
    ax.set_yticklabels(table.index, fontsize=8)
    ax.set_title(f"Standardized Loadings: {model_name}")
    fig.colorbar(image, ax=ax, fraction=0.04, pad=0.03)
    fig.tight_layout()
    fig.savefig(path, dpi=220)
    plt.close(fig)


def save_factor_corr_heatmap(corrs, model_name, path):
    if corrs.empty or "std_factor_correlation" not in corrs.columns:
        return
    data = corrs[corrs["model"].eq(model_name)].copy()
    if data.empty:
        return
    data["pair"] = data["lval"].astype(str) + " ~~ " + data["rval"].astype(str)
    data["case_subset"] = data["case"].astype(str) + " | " + data["subset"].astype(str)
    table = data.pivot_table(index="case_subset", columns="pair", values="std_factor_correlation", aggfunc="mean")
    if table.empty:
        return
    fig, ax = plt.subplots(figsize=(max(8, 1.2 * len(table.columns)), max(4, 0.42 * len(table))))
    image = ax.imshow(table.values, vmin=-1, vmax=1, cmap="RdBu_r", aspect="auto")
    ax.set_xticks(range(len(table.columns)))
    ax.set_xticklabels(table.columns, rotation=35, ha="right", fontsize=8)
    ax.set_yticks(range(len(table.index)))
    ax.set_yticklabels(table.index, fontsize=8)
    ax.set_title(f"Factor Correlations: {model_name}")
    fig.colorbar(image, ax=ax, fraction=0.04, pad=0.03)
    fig.tight_layout()
    fig.savefig(path, dpi=220)
    plt.close(fig)


PLOT_DIR = ONE_CLICK_OUTPUT_DIR / "plots"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

save_metric_barplot(fit_all, "RMSEA", PLOT_DIR / "model_comparison_rmsea.png", "CFA Model Comparison: RMSEA (lower is better)")
save_metric_barplot(fit_all, "CFI", PLOT_DIR / "model_comparison_cfi.png", "CFA Model Comparison: CFI (higher is better)")
save_metric_barplot(fit_all, "TLI", PLOT_DIR / "model_comparison_tli.png", "CFA Model Comparison: TLI (higher is better)")
save_metric_barplot(fit_all, "BIC", PLOT_DIR / "model_comparison_bic.png", "CFA Model Comparison: BIC (lower is better)")

for model_name in ["one_factor", "two_factor", "three_factor", "four_factor_sensitivity"]:
    save_loading_heatmap_table(load_all, model_name, PLOT_DIR / f"loadings_{model_name}.png")
    save_factor_corr_heatmap(corr_all, model_name, PLOT_DIR / f"factor_correlations_{model_name}.png")

print("Saved one-click tables to:", ONE_CLICK_OUTPUT_DIR.resolve())
print("Saved plots to:", PLOT_DIR.resolve())

## After CFA / 后续还能做什么

CFA is not the endpoint. It gives you a measurement model that can support deeper validity analyses.

### 1. Measurement invariance / 测量不变性

Question: Does the same CFA model measure the same construct across groups?

Groups to test:

- target model: Qwen vs Llama
- CALE variant: generic vs attack-aware vs full attack-aware
- framing style: neutral vs adversarial framing, if available
- dataset: FEVER vs Climate-FEVER / SciFact, if available

Claim if supported:

> The measurement structure showed preliminary invariance across target models, supporting comparisons of CALE diagnostic dimensions across model families.

### 2. Factor scores / latent dimension scores

Question: Can we use latent factor scores instead of only `final_score`?

Useful analyses:

- Which CALE variant improves `factual_handling` most?
- Which target model is weakest on `boundary_control`?
- Are NEI failures mainly boundary-control failures?
- Does adversarial framing reduce `adversarial_resistance` more than `factual_handling`?

Claim if useful:

> CFA-derived factor scores enabled dimension-specific diagnosis of evaluator behavior that would be obscured by a single aggregate score.

### 3. Criterion validity / 外部效标效度

Question: Do latent factors predict meaningful downstream failures or successes?

Possible criteria:

- `nei_uncertainty_failure_proxy`
- `supports_status_failure_proxy`
- `refutes_correction_credit_proxy`
- perturbation score shift
- label flip rate under framing
- agreement with stronger evaluator or human labels, if available

Claim if supported:

> The latent dimensions were associated with theoretically relevant downstream behavior proxies, strengthening the validity argument for CALE as a diagnostic measurement procedure.

### 4. Ordinal CFA / 更严格的 CFA

Because many indicators are binary or bounded, a stronger follow-up is ordinal CFA with WLSMV in `lavaan`.

Conservative bridge sentence:

> The present Python CFA-style analysis should be followed by ordinal CFA using robust categorical estimators before making strong claims about psychometric validation.

## Thesis-Safe Reporting Language

Use conservative wording if the notebook results go into the thesis:

> We fit theory-guided CFA-style SEM models to CALE diagnostic indicators using `semopy`, treating the bounded diagnostic scores as continuous observed variables. The primary correlated-factor model was compared with one-factor and two-factor alternatives using approximate fit indices and standardized loadings. Because several indicators are binary or bounded in `[0, 1]`, these results are interpreted as preliminary internal-structure evidence rather than definitive ordinal CFA validation.

Recommended table outputs:

- `cfa_model_comparison.csv`: model, N, df, chi-square, CFI, TLI, RMSEA, AIC, BIC.
- `cfa_standardized_loadings.csv`: factor, indicator, raw loading, standardized loading, standard error if available.
- `cfa_robustness_fit_indices.csv`: same fit summaries across target/variant subsets.
- `validity/cfa_claim_readiness.csv`: thesis-facing status, evidence, limitations, and safe wording for each psychometric claim.

- `validity/paper_cfa_claim_readiness.csv`: paper-facing compact claim-readiness table for thesis writing.


## Paper-Ready CFA Claim Summary / 论文可引用 CFA 结论表

This final export compresses the conservative claim-readiness table into paper-ready wording. It keeps `claim_area` and `status`, but rewrites `safe_wording` as short thesis sentences that can be quoted directly without overstating the psychometric evidence.

这一节把前面的 claim-readiness table 压缩成论文可直接引用的版本。它保留 `claim_area` 和 `status`，但把 `safe_wording` 改成短句，避免把 preliminary / screening evidence 写成 definitive validation。


In [ ]:
PAPER_CFA_CLAIM_READINESS_OUTPUT = VALIDITY_OUTPUT_DIR / "paper_cfa_claim_readiness.csv"

PAPER_SAFE_WORDING = {
    "internal_structure": "CALE indicators provide preliminary evidence of interpretable internal structure, especially around factual handling and adversarial resistance.",
    "convergent_validity": "Construct-relevant indicators show screening-level convergence with related diagnostic signals, but convergent validity is not yet confirmed.",
    "discriminant_validity": "Discriminant validity remains provisional because stronger nuisance-sensitivity tests are still needed.",
    "measurement_invariance": "Target- and backend-specific fits are robustness checks rather than formal measurement-invariance evidence.",
    "compact_capability_score": "Compact CALE dimensions should be treated as exploratory diagnostic summaries under a fixed evaluator backend and protocol.",
}

PAPER_STATUS_LABEL = {
    "supported_preliminarily": "preliminary support",
    "screening_evidence_only": "screening evidence only",
    "needs_stronger_nuisance_tests": "requires stronger nuisance tests",
    "not_fully_tested": "not fully tested",
    "exploratory": "exploratory",
    "not_evaluated_yet": "not evaluated yet",
}


def _paper_claim_source_table():
    """Return the existing conservative claim-readiness table without refitting models."""
    existing = globals().get("claim_readiness", pd.DataFrame())
    if isinstance(existing, pd.DataFrame) and not existing.empty:
        return existing.copy()
    if "CLAIM_READINESS_OUTPUT" in globals() and CLAIM_READINESS_OUTPUT.exists():
        return pd.read_csv(CLAIM_READINESS_OUTPUT)
    if "build_claim_readiness_table" in globals():
        return build_claim_readiness_table()
    return pd.DataFrame(columns=["claim_area", "status", "safe_wording"])


paper_cfa_claim_readiness = _paper_claim_source_table()
if paper_cfa_claim_readiness.empty:
    print("No CFA claim-readiness table is available yet. Run the previous claim-readiness cell first.")
else:
    paper_cfa_claim_readiness = paper_cfa_claim_readiness.copy()
    paper_cfa_claim_readiness["safe_wording"] = paper_cfa_claim_readiness["claim_area"].map(PAPER_SAFE_WORDING).fillna(
        paper_cfa_claim_readiness.get("safe_wording", "")
    )
    paper_cfa_claim_readiness["status_label"] = paper_cfa_claim_readiness["status"].map(PAPER_STATUS_LABEL).fillna(
        paper_cfa_claim_readiness["status"].astype(str).str.replace("_", " ")
    )

    output_cols = ["claim_area", "status", "status_label", "safe_wording"]
    optional_cols = ["supporting_outputs", "recommended_next_step"]
    output_cols.extend([c for c in optional_cols if c in paper_cfa_claim_readiness.columns])
    paper_cfa_claim_readiness = paper_cfa_claim_readiness[output_cols]

    PAPER_CFA_CLAIM_READINESS_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
    paper_cfa_claim_readiness.to_csv(PAPER_CFA_CLAIM_READINESS_OUTPUT, index=False)
    print("Saved paper-ready CFA claim summary to:", PAPER_CFA_CLAIM_READINESS_OUTPUT.resolve())
    display(paper_cfa_claim_readiness)
